## <center style="color:blue;">**FootVerse**</center>

### <center>**Modélisation et Analyse de Données Footballistiques**</center>

Ce projet a pour objectif de collecter des données de football à l’aide du web scraping (Selenium), puis de les transformer et nettoyer afin d’assurer leur qualité. Les données seront ensuite modélisées et stockées dans une base de données PostgreSQL. Enfin, un modèle de machine learning sera entraîné pour prédire l’équipe gagnante d’un match.

In [17]:
import os
import shutil

src = "../data/bronze/"
dest = "../data/silver/"

shutil.copy(os.path.join(src,"saisons.csv"), os.path.join(dest,"saisons.csv"))

shutil.copy(os.path.join(src,"teams.csv"), os.path.join(dest,"teams.csv"))

print("Fichiers Copiés avec Succés !")

Fichiers Copiés avec Succés !


<br>

### <span style="color:green;">**Gérer les Tables des Joueurs :**</span>

#### <span style="color:orange;">**1. Gérer les Valeurs Manquantes :**</span>

In [18]:
import pandas as pd

bronze_path = "../data/bronze/teams"

equipes = []

# Récupérer les noms des Equipes
for team in os.listdir(bronze_path) :
    equipes.append(team)

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "players.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    players_df = pd.read_csv(equipe_path)

    # Identifier le nombre des valeurs manquantes
    val_nulls = players_df.isnull().sum()
    print(f"- Nombre de Valeurs Manquantes dans l'Equipe '{equipe}' est : \n{val_nulls}")

    # Remplacer les lignes (axis=0) contenant les valeurs manquantes avec 0
    players_df = players_df.fillna(0)

    silver_path = "../data/silver"

    os.makedirs(os.path.join(silver_path, "teams", equipe), exist_ok=True)

    dest_path = f"../data/silver/teams/{equipe}"

    # Enregistrer DataFrame sous format csv
    players_df.to_csv(os.path.join(dest_path, "players.csv"), index=False)


- Nombre de Valeurs Manquantes dans l'Equipe 'Arsenal' est : 
Player     0
Nation     0
Pos        0
Age        0
MP         0
Starts     0
Min       13
90s       13
Gls       13
Ast       13
G+A       13
G-PK      13
PK        13
PKatt     13
CrdY      13
CrdR      13
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Aston Villa' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       6
90s       6
Gls       6
Ast       6
G+A       6
G-PK      6
PK        6
PKatt     6
CrdY      6
CrdR      6
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Bournemouth' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       9
90s       9
Gls       9
Ast       9
G+A       9
G-PK      9
PK        9
PKatt     9
CrdY      9
CrdR      9
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Brentford' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       7
90s       7
Gls       7
Ast

#### <span style="color:orange;">**2. Gérer les Doublons :**</span>

In [19]:
silver_path = "../data/silver/teams"

nulls = []

for equipe in equipes :
    equipe_path = os.path.join(silver_path, equipe, "players.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    players_df = pd.read_csv(equipe_path)

    # Identifier le nombre des valeurs manquantes
    val_nulls = players_df.duplicated().sum()
    

    nulls.append(val_nulls)

if sum(nulls) == 0 :
    print(f"Toutes les Equipes ne contiennent pas des Doublons !")

Toutes les Equipes ne contiennent pas des Doublons !


<br>

### <span style="color:green;">**Gérer les Tables des Statistiques des Matchs :**</span>

#### <span style="color:orange;">**1. Gérer les Valeurs Manquantes :**</span>

##### **1.1. Récupérer les Noms des Equipes :**

In [20]:
import pandas as pd

silver_path = "../data/silver/teams"

equipes = []

# Récupérer les noms des Equipes
for team in os.listdir(silver_path) :
    equipes.append(team)

##### **1.2. Identifier les Colonnes qui Contiennent des Valeurs Manquantes :**

In [21]:
import pandas as pd

bronze_path = "../data/bronze/teams"

null_colonnes = []

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "scores.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    scores_df = pd.read_csv(equipe_path)

    # Identifier tous les colonnes contenant des valeurs manquantes
    colonnes = list(scores_df.columns[scores_df.isnull().any()])

    # Stocker les colonnes 
    for col in colonnes :
        null_colonnes.append(col)

# Eliminer les colonnes doublons
null_cols = list(set(null_colonnes))

null_cols
    

['xG', 'Poss', 'xGA', 'Attendance', 'Referee']

##### **1.3. Remplacer les Valeurs Manquantes de chaque Colonne par la valeur convenable :**

In [22]:
import numpy as np

silver_path = "../data/silver/teams"

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "scores.csv")

    scores_df = pd.read_csv(equipe_path)

    cols = list(scores_df.columns[scores_df.isnull().any()])

    for col in cols :
        if col in ['xG','xGA'] :
            scores_df[col] = scores_df[col].fillna(0)
        elif col == 'Poss':
            mean_col = f"{scores_df[col].mean():.2f}"
            scores_df[col] = scores_df[col].fillna(mean_col)
        elif col == 'Referee':
            scores_df[col] = scores_df[col].fillna("Inconnu")
        elif col == 'Attendance' :
            scores_df[col] = scores_df[col].apply(lambda x : int(x.replace(",", "")) if isinstance(x, str) else np.nan )
            mean_col = f"{scores_df[col].mean():.2f}"
            scores_df[col] = scores_df[col].fillna(mean_col)
    
    scores_df.to_csv(os.path.join(silver_path,equipe,"scores.csv"), index=False)

    print(f"- {equipe} : Done !")

print("Valeurs Manquées Remplacées avec Succés !")


- Arsenal : Done !
- Aston Villa : Done !
- Bournemouth : Done !
- Brentford : Done !
- Brighton : Done !
- Chelsea : Done !
- Crystal Palace : Done !
- Everton : Done !
- Fulham : Done !
- Ipswich Town : Done !
- Leicester City : Done !
- Liverpool : Done !
- Manchester City : Done !
- Manchester Utd : Done !
- Newcastle Utd : Done !
- Nott'ham Forest : Done !
- Southampton : Done !
- Tottenham : Done !
- West Ham : Done !
- Wolves : Done !
Valeurs Manquées Remplacées avec Succés !


#### <span style="color:orange;">**2. Gérer les Doublons :**</span>

##### **2.1. Copier les Fichiers CSV des Statistiques des Matchs de Chaque Equipe en ajoutant la colonne ``SQUAD`` :**

In [ ]:
colonnes = []

for equipe in equipes :
    scores_df = pd.read_csv(os.path.join(silver_path, equipe, "scores.csv"))

    cols = list(scores_df.columns)

    scores_df["Squad"] = equipe

    colonnes = new_cols = [cols[0], "Squad"] + cols[1::]

    scores_df = scores_df[new_cols]

    scores_df.to_csv(os.path.join(silver_path, equipe, "scores.csv"), index=False)


##### **2.2. Créer un fichier CSV combinant tous les Matchs des Equipes :**

In [24]:
dfs = []

for equipe in equipes:
    path = os.path.join(silver_path, equipe, "scores.csv")
    scores_df = pd.read_csv(path)
    dfs.append(scores_df)        

df = pd.concat(dfs, ignore_index=True)

df.to_csv("../data/silver/scores.csv", index=False)

##### **2.3. Supprimer les Doublons dans le fichier CSV global des Statistiques de tous les Matches :**

In [41]:
df = pd.read_csv("../data/silver/scores.csv")

df["Resume"] = df.apply(
    lambda x : f"{x['Date']}_{x['Time']}_"+"_".join(sorted([x["Squad"], x["Opponent"]])),
    axis=1
)

df = df.drop_duplicates(subset="Resume")

df.to_csv("../data/silver/scores.csv", index=False)
